# ============================================================
# Étude épidémiologique : Analyse propagation virale (COVID-19)
# ============================================================

In [ ]:
# Imports des Modules

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import plotly.express as px


# -----------------------------
# 1. Chargement des données
# -----------------------------

In [ ]:
# URL officielle des données Google COVID-19 Open Data
url = "https://storage.googleapis.com/covid19-open-data/v3/epidemiology.csv"

# Charger directement sans enregistrer en local
usecols=["date","location_key","new_confirmed","cumulative_confirmed","new_deceased"]
df = pd.read_csv(url, usecols=usecols, parse_dates=["date"])

df.head()



In [ ]:
# Nombre de pays
print("Nombre de pays/territoires:", df["location_key"].nunique()) #?

# Dates min / max
print("Période:", df["date"].min(), "→", df["date"].max())


# -----------------------------
# 2. Exploitation et affichage des données
# -----------------------------

In [ ]:
country = "FR"   # code ISO pour la France
df_country = df[df["location_key"] == country].copy()

df_country.set_index("date")[["new_confirmed"]].plot(figsize=(12,4), title=f"Nouveaux cas journaliers - {country}")
plt.show()

df_country.set_index("date")[["cumulative_confirmed"]].plot(figsize=(12,4), title=f"Cas cumulés - {country}")
plt.show()


# -----------------------------
# 3. Interprétation des données
#    Explication des ourtils de mesure
# -----------------------------

R₀ (taux de reproduction de base)
→ C’est une constante théorique qui décrit au tout début d’une épidémie le nombre moyen de personnes infectées par un individu malade dans une population totalement susceptible (aucune immunité, aucun vaccin, aucune mesure de contrôle).
Exemple : si R₀ = 3, une personne infectée contamine en moyenne 3 autres.

Rₜ (taux de reproduction effectif, ou instantané)
→ C’est une version dynamique de R₀ : il évolue dans le temps en fonction de l’immunité acquise, des mesures sanitaires, des comportements sociaux, etc.
Exemple : si R₀ = 3 mais que 50% de la population est immunisée, alors Rₜ sera inférieur à 3.

In [ ]:
# Logarithme des cas cumulés
df_country["log_cases"] = np.log1p(df_country["cumulative_confirmed"])

# Croissance journalière
df_country["growth_rate"] = df_country["log_cases"].diff()

# Estimation simple de R_t (avec période infectieuse moyenne de 5 jours)
infectious_period = 5
df_country["R_t"] = 1 + df_country["growth_rate"] * infectious_period

df_country.plot(x="date", y="R_t", figsize=(12,4), title=f"Estimation Rₜ - {country}")
plt.axhline(1, color="red", linestyle="--")
plt.show()


# -----------------------------
# 3.1 Création du dictionnaire ISO-2 -> ISO-3
# -----------------------------

In [ ]:
# Mapping ISO-2 → ISO-3 pour tous les pays
iso2_to_iso3 = {
    "AD": "AND","AE": "ARE","AF": "AFG","AG": "ATG","AI": "AIA","AL": "ALB","AM": "ARM",
    "AO": "AGO","AR": "ARG","AT": "AUT","AU": "AUS","AW": "ABW","AX": "ALA","AZ": "AZE",
    "BA": "BIH","BB": "BRB","BD": "BGD","BE": "BEL","BF": "BFA","BG": "BGR","BH": "BHR",
    "BI": "BDI","BJ": "BEN","BL": "BLM","BM": "BMU","BN": "BRN","BO": "BOL","BQ": "BES",
    "BR": "BRA","BS": "BHS","BT": "BTN","BV": "BVT","BW": "BWA","BY": "BLR","BZ": "BLZ",
    "CA": "CAN","CC": "CCK","CD": "COD","CF": "CAF","CG": "COG","CH": "CHE","CI": "CIV",
    "CK": "COK","CL": "CHL","CM": "CMR","CN": "CHN","CO": "COL","CR": "CRI","CU": "CUB",
    "CV": "CPV","CW": "CUW","CX": "CXR","CY": "CYP","CZ": "CZE","DE": "DEU","DJ": "DJI",
    "DK": "DNK","DM": "DMA","DO": "DOM","DZ": "DZA","EC": "ECU","EE": "EST","EG": "EGY",
    "EH": "ESH","ER": "ERI","ES": "ESP","ET": "ETH","FI": "FIN","FJ": "FJI","FK": "FLK",
    "FM": "FSM","FO": "FRO","FR": "FRA","GA": "GAB","GB": "GBR","GD": "GRD","GE": "GEO",
    "GF": "GUF","GG": "GGY","GH": "GHA","GI": "GIB","GL": "GRL","GM": "GMB","GN": "GIN",
    "GP": "GLP","GQ": "GNQ","GR": "GRC","GT": "GTM","GU": "GUM","GW": "GNB","GY": "GUY",
    "HK": "HKG","HM": "HMD","HN": "HND","HR": "HRV","HT": "HTI","HU": "HUN","ID": "IDN",
    "IE": "IRL","IL": "ISR","IM": "IMN","IN": "IND","IO": "IOT","IQ": "IRQ","IR": "IRN",
    "IS": "ISL","IT": "ITA","JE": "JEY","JM": "JAM","JO": "JOR","JP": "JPN","KE": "KEN",
    "KG": "KGZ","KH": "KHM","KI": "KIR","KM": "COM","KN": "KNA","KP": "PRK","KR": "KOR",
    "KW": "KWT","KY": "CYM","KZ": "KAZ","LA": "LAO","LB": "LBN","LC": "LCA","LI": "LIE",
    "LK": "LKA","LR": "LBR","LS": "LSO","LT": "LTU","LU": "LUX","LV": "LVA","LY": "LBY",
    "MA": "MAR","MC": "MCO","MD": "MDA","ME": "MNE","MF": "MAF","MG": "MDG","MH": "MHL",
    "MK": "MKD","ML": "MLI","MM": "MMR","MN": "MNG","MO": "MAC","MP": "MNP","MQ": "MTQ",
    "MR": "MRT","MS": "MSR","MT": "MLT","MU": "MUS","MV": "MDV","MW": "MWI","MX": "MEX",
    "MY": "MYS","MZ": "MOZ","NA": "NAM","NC": "NCL","NE": "NER","NF": "NFK","NG": "NGA",
    "NI": "NIC","NL": "NLD","NO": "NOR","NP": "NPL","NR": "NRU","NU": "NIU","NZ": "NZL",
    "OM": "OMN","PA": "PAN","PE": "PER","PF": "PYF","PG": "PNG","PH": "PHL","PK": "PAK",
    "PL": "POL","PM": "SPM","PN": "PCN","PR": "PRI","PT": "PRT","PW": "PLW","PY": "PRY",
    "QA": "QAT","RE": "REU","RO": "ROU","RS": "SRB","RU": "RUS","RW": "RWA","SA": "SAU",
    "SB": "SLB","SC": "SYC","SD": "SDN","SE": "SWE","SG": "SGP","SH": "SHN","SI": "SVN",
    "SJ": "SJM","SK": "SVK","SL": "SLE","SM": "SMR","SN": "SEN","SO": "SOM","SR": "SUR",
    "SS": "SSD","ST": "STP","SV": "SLV","SX": "SXM","SY": "SYR","SZ": "SWZ","TC": "TCA",
    "TD": "TCD","TF": "ATF","TG": "TGO","TH": "THA","TJ": "TJK","TK": "TKL","TL": "TLS",
    "TM": "TKM","TN": "TUN","TO": "TON","TR": "TUR","TT": "TTO","TV": "TUV","TZ": "TZA",
    "UA": "UKR","UG": "UGA","UM": "UMI","US": "USA","UY": "URY","UZ": "UZB","VA": "VAT",
    "VC": "VCT","VE": "VEN","VG": "VGB","VI": "VIR","VN": "VNM","VU": "VUT","WF": "WLF",
    "WS": "WSM","YE": "YEM","YT": "MYT","ZA": "ZAF","ZM": "ZMB","ZW": "ZWE"
}

In [ ]:
# Convertir la date en datetime si ce n'est pas déjà fait
df["date"] = pd.to_datetime(df["date"])

# Extraire le code ISO-2 pour toutes les lignes (2 premières lettres)
df["iso2"] = df["location_key"].str[:2]

# Faire le mapping ISO-2 → ISO-3
df["iso_alpha3"] = df["iso2"].map(iso2_to_iso3)

# Supprimer les lignes non reconnues
df = df.dropna(subset=["iso_alpha3"])

# Grouper par pays ET date pour avoir le cumul au niveau pays
df_country_date = df.groupby(["iso_alpha3", "date"], as_index=False)["cumulative_confirmed"].sum()

# Comparaison pays avant et après regroupement par zone et tranformation en iso-3
#print(df_country_date["iso_alpha3"].unique())

# Pour chaque pays, récupérer la dernière valeur disponible
latest_list = []

for cnt in df_country_date["iso_alpha3"].unique():
    df_temp = df_country_date[df_country_date["iso_alpha3"] == cnt]
    idx = df_temp["date"].idxmax()  # l'index de la dernière date pour ce pays
    latest_list.append(df_temp.loc[idx])

# Concaténer toutes les dernières lignes dans un seul DataFrame
df_latest = pd.DataFrame(latest_list).reset_index(drop=True)

#print(df_latest["iso_alpha3"].unique())  # devrait contenir tous les pays


# -----------------------------
# 3.2 Création de la carte des cas cumulés
# -----------------------------

In [ ]:
# Affichage via valeur relative (% de la valeur maximale)
df_latest["cumulative_percent"] = df_latest["cumulative_confirmed"] / df_latest["cumulative_confirmed"].max() * 100

fig_pct = px.choropleth(
    df_latest,
    locations="iso_alpha3",
    color="cumulative_percent",
    hover_name="iso_alpha3",
    color_continuous_scale="Reds",
    title=f"Cas cumulés COVID-19 (% de la valeur max) au {df_latest['date'].max()}",
    projection="natural earth"
)
fig_pct.update_coloraxes(colorbar_title="% du max")
fig_pct.show()


In [ ]:
df_country.set_index("date")[["new_deceased"]].plot(figsize=(12,4), title=f"Nouveaux décès journaliers - {country}")
plt.show()


# -----------------------------
# 3.3 Comparaison des données
# -----------------------------

In [ ]:
# Comparer plusieurs pays
countries = ["FR", "IT", "ES"]
df_sel = df[df["location_key"].isin(countries)]

plt.figure(figsize=(12,5))
for c in countries:
    plt.plot(df_sel[df_sel["location_key"]==c]["date"],
             df_sel[df_sel["location_key"]==c]["cumulative_confirmed"],
             label=c)
plt.legend()
plt.title("Comparaison cas cumulés sur une période")
plt.show()

plt.figure(figsize=(12,5))
for c in countries:
    plt.plot(df_sel[df_sel["location_key"]==c]["date"],
             df_sel[df_sel["location_key"]==c]["new_deceased"],
             label=c)
plt.legend()
plt.title("Comparaison des nouvaux cas sur une période")
plt.show()


# 📊 Synthèse – Étude épidémiologique

## Objectifs
- Charger et explorer les données COVID-19 (Google Open Data)
- Étudier l’évolution temporelle des cas
- Estimer la dynamique de propagation par le calcul de Rₜ
- Visualiser les résultats sur une carte mondiale

## Méthodologie
1. Chargement du dataset compressé (.csv.gz) via `pandas`
2. Sélection d’un pays (exemple : France) pour les analyses détaillées
3. Calcul du taux de croissance exponentielle à partir des cas cumulés
4. Estimation du taux de reproduction effectif **Rₜ**
   - Hypothèse : période infectieuse moyenne = 5 jours
5. Visualisation :
   - Courbes temporelles (cas journaliers, cumulés, décès)
   - Estimation de Rₜ dans le temps
   - Carte mondiale des cas cumulés à la dernière date
   - Comparaison entre pays sélectionnés sur différentes catégories: "new-deceased", "cumulative_confirmed"

## Résultats
- Les courbes temporelles montrent des vagues épidémiques distinctes
- L’estimation de **Rₜ** met en évidence des phases > 1 (propagation) et < 1 (contrôle)
- La carte mondiale illustre les différences régionales marquées dans la propagation

## Commentaires
- L’estimation de Rₜ est sensible aux données (sous-détection, délais de reporting…)
- R₀ est une valeur théorique initiale, tandis que notre étude porte sur **Rₜ** observé
- Les données Google sont très riches : possibilité d’étendre l’étude à la vaccination, mobilité ou politiques de santé publique, ou même de faire une étude plus poussée par zone
